# Assignment 3: UAV Drone Detection & Tracking

**SAHI sliced inference + SORT multi-object Kalman tracking**

This notebook runs the full pipeline on a Colab GPU.

| Step | What it does | Expected time |
|------|-------------|---------------|
| 0 | Verify GPU | instant |
| 1 | Install deps | ~1 min |
| 2 | Upload project zip | ~1 min |
| 3 | Verify setup | ~10 sec |
| 4 | Quick test (5 frames) | ~1 min |
| 5 | Full pipeline (6816 frames) | ~15-30 min |
| 6-8 | Review + download | ~2 min |

## 0. Verify GPU

If this fails: **Runtime > Change runtime type > T4 GPU**

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Go to: Runtime > Change runtime type > T4 GPU\n"
        "Then re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"Python: {__import__('sys').version}")
print("OK")

## 1. Install dependencies

Installs ultralytics, SAHI, filterpy, and scipy. Uses Colab's pre-installed OpenCV (no headless conflict).

In [ ]:
# Pin pillow to avoid ultralytics import crash on Colab (pillow 12.x breaks PIL._typing)
# Do NOT install opencv-python-headless — Colab already has opencv-python
!pip install -q "pillow>=7.1.2,<12.0.0" ultralytics filterpy sahi scipy tqdm pyarrow datasets

# Verify critical imports
import ultralytics
import sahi
import scipy
import cv2
print(f"ultralytics {ultralytics.__version__}")
print(f"sahi {sahi.__version__}")
print(f"scipy {scipy.__version__}")
print(f"opencv {cv2.__version__}")
print("All deps OK")

## 2. Upload project files

**Option A** (recommended for large files): Copy `colab_upload.zip` to your Google Drive first, then mount.

**Option B**: Direct browser upload (works for 160MB, may be slow on bad connections).

**Run ONE of the two cells below, not both.**

In [ ]:
# === OPTION A: Google Drive (recommended) ===
# 1. Upload colab_upload.zip to your Google Drive root first
# 2. Then run this cell

from google.colab import drive
import shutil
import zipfile
import os

drive.mount("/content/drive")

DRIVE_ZIP = "/content/drive/MyDrive/colab_upload.zip"
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(
        f"{DRIVE_ZIP} not found.\n"
        "Upload colab_upload.zip to the ROOT of your Google Drive, then re-run."
    )

WORK = "/content/drone_pipeline"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)
os.chdir(WORK)

print(f"Copying from Drive ({os.path.getsize(DRIVE_ZIP) / 1e6:.0f} MB)...")
shutil.copy2(DRIVE_ZIP, "colab_upload.zip")

with zipfile.ZipFile("colab_upload.zip", "r") as z:
    z.extractall(".")

# Verify extraction
for f in ["main.py", "pipeline.py", "tracker.py", "best.pt"]:
    assert os.path.exists(f), f"Missing: {f}"
assert os.path.isdir("videos"), "Missing: videos/"

print("Extracted and verified:")
!ls -lh *.py best.pt && ls -lh videos/

In [ ]:
# === OPTION B: Direct browser upload ===
# Run this cell ONLY if you skipped Option A above

from google.colab import files
import zipfile
import os
import shutil

WORK = "/content/drone_pipeline"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
os.makedirs(WORK)
os.chdir(WORK)

print("Select colab_upload.zip from your computer...")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"Received: {zip_name} ({len(uploaded[zip_name]) / 1e6:.0f} MB)")

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(".")

# Verify extraction
for f in ["main.py", "pipeline.py", "tracker.py", "best.pt"]:
    assert os.path.exists(f), f"Missing after extraction: {f}"
assert os.path.isdir("videos"), "Missing: videos/ directory"

print("\nExtracted and verified:")
!ls -lh *.py best.pt && ls -lh videos/

## 3. Verify setup

In [ ]:
import subprocess
import sys
sys.path.insert(0, "/content/drone_pipeline")

import torch
from ultralytics import YOLO
from sahi import AutoDetectionModel

# Verify model loads
model = YOLO("best.pt")
print(f"Model classes: {model.names}")
print(f"CUDA device:   {torch.cuda.get_device_name(0)}")

# Verify SAHI can wrap the model
sahi_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="best.pt",
    confidence_threshold=0.10,
    device="cuda:0",
)
print(f"SAHI model:    loaded on {sahi_model.device}")

# Verify our custom modules import
print("Custom modules: OK")

# Verify ffmpeg
r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print(f"ffmpeg:        {r.stdout.splitlines()[0] if r.returncode == 0 else 'NOT FOUND'}")

del model, sahi_model
torch.cuda.empty_cache()
print("\nAll checks passed.")

## 4. Quick test (5 frames per video)

Runs the full pipeline on just 5 frames to verify everything works end-to-end before committing to the full run.

In [ ]:
!python main.py track --weights best.pt --device cuda --max-frames 5 --overwrite-outputs 2>&1

In [ ]:
import os
import glob

# Check outputs exist
det_frames = sorted(glob.glob("detections/*.jpg"))
out_videos = sorted(glob.glob("output_videos/*.mp4"))

print(f"Detection frames: {len(det_frames)}")
print(f"Output videos:    {len(out_videos)}")

if not det_frames:
    print("WARNING: No detection frames produced. Check the output above for errors.")
else:
    # Show first detection
    from IPython.display import display, Image as IPImage
    print(f"\nSample: {det_frames[0]}")
    display(IPImage(filename=det_frames[0], width=640))
    print("\nQuick test passed. Safe to run full pipeline.")

## 5. Full pipeline run

Processes all frames from both videos with SAHI + SORT. On a T4 GPU this takes ~15-30 minutes.

In [ ]:
# Clean outputs from quick test
import shutil
import os
for d in ["detections", "output_videos"]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Keep frames/ — no need to re-extract
print("Cleaned detection and output directories.")
print("Starting full pipeline...\n")

In [ ]:
%%time
!python main.py track --weights best.pt --device cuda --overwrite-outputs 2>&1

## 6. Review results

In [ ]:
import json
from pathlib import Path

summary_path = Path("artifacts/pipeline_summary.json")
if not summary_path.exists():
    raise FileNotFoundError("pipeline_summary.json not found — the pipeline may have crashed. Check cell 5 output.")

summary = json.loads(summary_path.read_text())
for s in summary:
    rate = s['detection_frames'] / s['sampled_frames'] * 100 if s['sampled_frames'] else 0
    print(f"\n{s['video_name']}:")
    print(f"  Sampled frames:   {s['sampled_frames']}")
    print(f"  Detection frames: {s['detection_frames']} ({rate:.1f}%)")
    print(f"  Rendered frames:  {s['rendered_frames']}")
    print(f"  Output video:     {s['output_video']}")

In [ ]:
# Preview detection frames (first, middle, last from each video)
from IPython.display import display, Image as IPImage
import glob

for video_name in ["drone_video_1", "drone_video_2"]:
    det_frames = sorted(glob.glob(f"detections/{video_name}_*.jpg"))
    print(f"\n{video_name}: {len(det_frames)} detection frames")
    if det_frames:
        for i in [0, len(det_frames) // 2, -1]:
            print(f"  {det_frames[i]}")
            display(IPImage(filename=det_frames[i], width=640))
    else:
        print("  (no detections)")

In [ ]:
# Play output videos inline
# NOTE: Large videos may not embed well. If playback fails, download them in step 8.
from IPython.display import HTML, display
from base64 import b64encode
import os
import glob

for vpath in sorted(glob.glob("output_videos/*.mp4")):
    size_mb = os.path.getsize(vpath) / 1e6
    print(f"\n{vpath} ({size_mb:.1f} MB)")
    if size_mb > 50:
        print("  (too large for inline playback — download in step 8)")
        continue
    data = open(vpath, "rb").read()
    b64 = b64encode(data).decode()
    display(HTML(f'<video width="720" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

## 7. Build Parquet dataset

In [ ]:
!python main.py upload --detections-dir detections 2>&1

import os
parquet_path = "artifacts/detections.parquet"
if os.path.exists(parquet_path):
    print(f"\nParquet file: {os.path.getsize(parquet_path) / 1e6:.1f} MB")
else:
    print("WARNING: Parquet file was not created.")

In [ ]:
# === Push to HuggingFace Hub ===
# Uncomment all lines below to push. You will be prompted to log in.

# from huggingface_hub import login
# login()
# !python main.py upload --detections-dir detections --repo-id HarshAgarwalNYU/Assignment3Drone

## 8. Download outputs

In [ ]:
from google.colab import files
import glob
import os

# Download output videos
for vpath in sorted(glob.glob("output_videos/*.mp4")):
    size_mb = os.path.getsize(vpath) / 1e6
    print(f"Downloading {vpath} ({size_mb:.1f} MB)...")
    files.download(vpath)

# Download summary
if os.path.exists("artifacts/pipeline_summary.json"):
    files.download("artifacts/pipeline_summary.json")

# Download parquet
if os.path.exists("artifacts/detections.parquet"):
    files.download("artifacts/detections.parquet")

print("Done!")